**Navigation** : [jumeau Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb) · [jumeau Rust](Argumentation-03-Dung-AF-Semantics-Rust.ipynb)

# Argumentation abstraite de Dung — sémantiques grounded, preferred, stable

> **Notebook fondationnel** (non-agentique) de la série *ArgumentAnalysis*.
> Il précède et éclaire l'aperçu de [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb)
> et l'utilisation du solveur Tweety dans [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb).

## Pourquoi ce notebook

Le notebook [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) invoque la sémantique
*grounded* de Dung via le solveur Tweety (JVM) comme une boîte noire : on appelle
`SimpleGroundedReasoner`, on récupère une extension. Mais **que calcule exactement ce
solveur, et pourquoi existe-t-il plusieurs sémantiques concurrentes (grounded, preferred,
stable) qui ne donnent pas toujours le même verdict ?**

Ce notebook répond à cette question en **construisant les sémantiques de zéro**, en C#
standard (BCL .NET seule, sans dépendance externe), sans JVM ni solveur externe. L'objectif n'est pas
de remplacer Tweety (moins performant, non utilisable en production sur de grands graphes) mais
de **rendre transparent** ce que le solveur fait — de façon à ce que l'appel `getModels(theory)` du
notebook 2-formal cesse d'être magique.

**Référence** : Dung, *On the Acceptability of Arguments and its Fundamental Rôle in
Nonmonotonic Reasoning, Logic Programming and n-Person Games* (Artificial Intelligence,
1995). Le formalisme y est introduit dans sa forme la plus abstraite, indépendante de
toute structure interne des arguments.

## 1. Cadre d'argumentation abstrait (Dung 1995)

Un **abstract argumentation framework** (AF) est un couple $F = \langle A, R \rangle$ où :

- $A$ est un ensemble fini d'**arguments** (des symboles opaques — on ne s'intéresse pas à leur contenu interne, seulement à qui attaque qui) ;
- $R \subseteq A \times A$ est une relation d'**attaque** : $(a, b) \in R$ se lit « $a$ attaque $b$ ».

L'abstraction est délibérée : la théorie de Dung porte sur la *structure du débat*
(graphe d'attaque), pas sur le sens des arguments. Deux débats aux graphes identiques
reçoivent le même verdict d'acceptabilité, quelle que soit leur matière. C'est cette
abstraction qui rend les sémantiques calculables et comparables.

Implémentons le cadre :

In [1]:
// Illustration 1 — Cadre d'argumentation abstrait de Dung.

// Un petit exemple ludique pour démarrer : a et b s'attaquent mutuellement.
var af0 = new AF(
    new[] { "a", "b" },
    new[] { ("a", "b"), ("b", "a") });

Console.WriteLine(af0);
Console.WriteLine($"Attaquants de a : {{{string.Join(", ", af0.Attackers("a"))}}}");

// Contrôle exécutable : les attaquants de a sont exactement {b} (parité avec la source).
var attaquantsA = af0.Attackers("a");
var attackersOk = attaquantsA.Count == 1 && attaquantsA[0] == "b";
Console.WriteLine(
    $"  attendu {{b}} vs calculé {{{string.Join(", ", attaquantsA)}}} : {(attackersOk ? "CONFORME" : "ÉCART")}");
if (!attackersOk)
{
    throw new InvalidOperationException("Les attaquants de a doivent être exactement {b}.");
}

public sealed class AF
{
    public SortedSet<string> Args { get; }

    // ensemble de couples (attaquant, attaque)
    public HashSet<(string Attacker, string Target)> Attacks { get; }

    public AF(
        IEnumerable<string> args,
        IEnumerable<(string Attacker, string Target)> attacks)
    {
        Args = new SortedSet<string>(args);
        Attacks = new HashSet<(string Attacker, string Target)>(attacks);
    }

    public IReadOnlyList<string> Attackers(string x)
    {
        return Attacks
            .Where(attack => attack.Target == x)
            .Select(attack => attack.Attacker)
            .Distinct()
            .OrderBy(attacker => attacker, StringComparer.Ordinal)
            .ToArray();
    }

    public override string ToString()
    {
        var arguments = string.Join(", ", Args);
        var attacks = string.Join(
            ", ",
            Attacks
                .OrderBy(attack => attack.Attacker, StringComparer.Ordinal)
                .ThenBy(attack => attack.Target, StringComparer.Ordinal)
                .Select(attack => $"{attack.Attacker}->{attack.Target}"));

        return $"AF(args=[{arguments}], attacks=[{attacks}])";
    }
}

AF(args=[a, b], attacks=[a->b, b->a])


Attaquants de a : {b}


  attendu {b} vs calculé {b} : CONFORME


## 2. De la conflictualité à l'admissibilité

Toute la théorie repose sur une chaîne de définitions emboîtées. Chaque notion prépare
la suivante.

### 2.1 Sans conflit interne

Un ensemble $S$ est **sans conflit** (*conflict-free*) si aucun de ses membres n'en
attaque un autre : $\nexists a, b \in S$ tels que $(a, b) \in R$.

C'est la condition minimale de cohérence : un ensemble qui se contredit n'est pas
une position défendable.

In [2]:
// Illustration 2 — Sans conflit interne.

var both = new HashSet<string> { "a", "b" };
var onlyA = new HashSet<string> { "a" };
var onlyB = new HashSet<string> { "b" };

var cfBoth = ConflictFreeChecker.IsConflictFree(af0, both);
var cfOnlyA = ConflictFreeChecker.IsConflictFree(af0, onlyA);
var cfOnlyB = ConflictFreeChecker.IsConflictFree(af0, onlyB);

Console.WriteLine($"a,b se battent -> {{a,b}} sans conflit ? {cfBoth}");
Console.WriteLine($"  attendu False vs calculé {cfBoth} : {(cfBoth ? "ÉCART" : "CONFORME")}");
Console.WriteLine($"{{a}} seul  -> sans conflit ? {cfOnlyA}");
Console.WriteLine($"  attendu True vs calculé {cfOnlyA} : {(cfOnlyA ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"{{b}} seul  -> sans conflit ? {cfOnlyB}");
Console.WriteLine($"  attendu True vs calculé {cfOnlyB} : {(cfOnlyB ? "CONFORME" : "ÉCART")}");

if (cfBoth || !cfOnlyA || !cfOnlyB)
{
    throw new InvalidOperationException("Les cas de conflit interne ne correspondent pas à la source.");
}

public static class ConflictFreeChecker
{
    // True si aucun argument de S n'en attaque un autre dans S.
    public static bool IsConflictFree(AF af, IEnumerable<string> candidates)
    {
        var set = candidates.ToHashSet(StringComparer.Ordinal);
        return !af.Attacks.Any(attack =>
            set.Contains(attack.Attacker) && set.Contains(attack.Target));
    }
}

a,b se battent -> {a,b} sans conflit ? False


  attendu False vs calculé False : CONFORME


{a} seul  -> sans conflit ? True


  attendu True vs calculé True : CONFORME


{b} seul  -> sans conflit ? True


  attendu True vs calculé True : CONFORME


### 2.2 Défait et défendu

Pour aller plus loin que la simple absence de conflit, il faut formaliser la notion de
**défense** :

- $S$ **défait** un argument $x$ (noté $S \rightsquigarrow x$) si au moins un membre de $S$
  attaque $x$ : $\exists a \in S, (a, x) \in R$.
- $S$ **défend** un argument $x$ si $S$ défait *tous* les attaquants de $x$ :
  $\forall b$ tel que $(b, x) \in R$, $S$ défait $b$.

Autrement dit : $x$ est défendu par $S$ si toute attaque contre $x$ est contrée par une
attaque venant de $S$. C'est le cœur du raisonnement argumentatif — on accepte un
argument non parce qu'il est vrai, mais parce qu'on peut le *protéger*.

In [3]:
// Illustration 3 — Défait et défendu.

var onlyA = new HashSet<string> { "a" };

var defendsA = DefenseChecker.Defends(af0, onlyA, "a");
var defeatsB = DefenseChecker.Defeats(af0, onlyA, "b");
var defeatsA = DefenseChecker.Defeats(af0, onlyA, "a");

Console.WriteLine($"Dans af0, {{a}} défend-il a ? {defendsA}");
Console.WriteLine($"  attendu True vs calculé {defendsA} : {(defendsA ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"  (a est attaqué par b ; {{a}} défait-il b ? a->b : {defeatsB} )");
Console.WriteLine($"  attendu True vs calculé {defeatsB} : {(defeatsB ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"  (a n'est attaqué par aucun membre de {{a}} : {{a}} défait-il a ? {defeatsA} )");
Console.WriteLine($"  attendu False vs calculé {defeatsA} : {(defeatsA ? "ÉCART" : "CONFORME")}");

if (!defendsA || !defeatsB || defeatsA)
{
    throw new InvalidOperationException("Les résultats de défense/défaite ne correspondent pas à la source.");
}

public static class DefenseChecker
{
    // True si S défait x (au moins un membre de S attaque x).
    public static bool Defeats(AF af, IEnumerable<string> candidates, string x)
    {
        return candidates.Any(attacker => af.Attacks.Contains((attacker, x)));
    }

    // True si S défend x : tout attaquant de x est défait par S.
    public static bool Defends(AF af, IEnumerable<string> candidates, string x)
    {
        return af.Attackers(x).All(attacker => Defeats(af, candidates, attacker));
    }
}

Dans af0, {a} défend-il a ? True


  attendu True vs calculé True : CONFORME


  (a est attaqué par b ; {a} défait-il b ? a->b : True )


  attendu True vs calculé True : CONFORME


  (a n'est attaqué par aucun membre de {a} : {a} défait-il a ? False )


  attendu False vs calculé False : CONFORME


### 2.3 Admissible

Un ensemble $S$ est **admissible** s'il est sans conflit *et* si chacun de ses membres
est défendu par $S$ :

$$S \text{ admissible} \iff S \text{ sans conflit} \;\land\; \forall a \in S, \; S \text{ défend } a$$

L'admissibilité capture l'idée d'une position **auto-cohérente et auto-défendable** :
aucune contradiction interne, et chaque membre résiste aux attaques extérieures.

In [4]:
// Illustration 4 — Admissible.

// Cas testés : vide, {a}, {b}, {a b} — les mêmes que la source.
var casAdmissibilite = new (string Label, string[] Members, bool Expected)[]
{
    ("∅", Array.Empty<string>(), true),
    ("a", new[] { "a" }, true),
    ("b", new[] { "b" }, true),
    ("a b", new[] { "a", "b" }, false),
};

foreach (var (label, members, expected) in casAdmissibilite)
{
    var candidateSet = new HashSet<string>(members, StringComparer.Ordinal);
    var admissible = AdmissibilityChecker.IsAdmissible(af0, candidateSet);
    Console.WriteLine($"{{{label}}} admissible ? {admissible}");
    Console.WriteLine($"  attendu {expected} vs calculé {admissible} : {(admissible == expected ? "CONFORME" : "ÉCART")}");
    if (admissible != expected)
    {
        throw new InvalidOperationException($"Admissibilité inattendue pour {{{label}}}.");
    }
}

public static class AdmissibilityChecker
{
    // True si S est admissible : sans conflit et chacun de ses membres est défendu.
    public static bool IsAdmissible(AF af, IEnumerable<string> candidates)
    {
        var set = candidates.ToHashSet(StringComparer.Ordinal);
        if (!ConflictFreeChecker.IsConflictFree(af, set))
        {
            return false;
        }

        return set.All(argument => DefenseChecker.Defends(af, set, argument));
    }
}

{∅} admissible ? True


  attendu True vs calculé True : CONFORME


{a} admissible ? True


  attendu True vs calculé True : CONFORME


{b} admissible ? True


  attendu True vs calculé True : CONFORME


{a b} admissible ? False


  attendu False vs calculé False : CONFORME


## 3. Les trois sémantiques

L'admissibilité définit un *spectre* d'ensembles acceptables. Les **sémantiques**
sélectionnent, parmi les ensembles admissibles, ceux qui méritent d'être appelés
« extensions » (le verdict final). Dung en définit trois principales, qui encodent des
**attitudes de raisonnement distinctes** :

| Sémantique | Attitude | Définition |
|------------|----------|------------|
| **Grounded** | *sceptique* (le strict minimum certain) | L'unique extension complète minimale ; obtenue par point fixe : on part de $\emptyset$ et on ajoute itérativement tout argument défendu. |
| **Preferred** | *crédule* (le maximum défendable) | Les extensions complètes **maximales** (par inclusion) ; il peut y en avoir plusieurs, incompatibles entre elles. |
| **Stable** | *auto-suffisante* (elle statut sur tout) | Une extension admissible qui **défait tout argument extérieur** : $\forall x \notin S, S \rightsquigarrow x$. |
| **Semi-stable** *(§ 4bis)* | *pragmatique* (statuer au maximum) | Les admissibles qui maximisent la portée $S \cup S^+$ ; **existe toujours**, et coïncide avec la stable quand celle-ci existe. |
| **Idéale** *(§ 4bis)* | *sceptique maximale* (verdict unique) | La plus grande admissible contenue dans **toutes** les preferred ; unique, avec $\text{grounded} \subseteq \text{idéale} \subseteq \bigcap \text{preferred}$. |

Ces trois sémantiques ne sont pas redondantes : elles répondent à la question
« quels arguments accepter ? » sous trois régimes de prudence différents. Le théorème
central de Dung garantit l'inclusion **grounded $\subseteq$ preferred**, mais **stable
peut ne pas exister**, et **preferred peut compter plusieurs extensions incompatibles**.

Implémentons les trois par énumération (la théorie garantit que pour un AF fini, le nombre
d'extensions est fini ; l'énumération des $2^{|A|}$ sous-ensembles reste tractable pour
les petits graphes pédagogiques).

In [5]:
// Illustration 5 — Les trois sémantiques : grounded, preferred, stable.

public static class Semantics
{
    // Tous les sous-ensembles de args, par taille croissante.
    public static List<string[]> Powerset(IEnumerable<string> args)
    {
        var items = args.ToArray();
        var result = new List<string[]>();
        for (int size = 0; size <= items.Length; size++)
        {
            foreach (var combo in Combinations(items, 0, size))
            {
                result.Add(combo);
            }
        }

        return result;
    }

    private static IEnumerable<string[]> Combinations(string[] items, int start, int size)
    {
        if (size == 0)
        {
            yield return Array.Empty<string>();
            yield break;
        }

        for (int i = start; i <= items.Length - size; i++)
        {
            foreach (var rest in Combinations(items, i + 1, size - 1))
            {
                var combo = new string[size];
                combo[0] = items[i];
                Array.Copy(rest, 0, combo, 1, rest.Length);
                yield return combo;
            }
        }
    }

    // Extension grounded : point fixe à partir de l'ensemble vide.
    // On ajoute itérativement tout argument défendu par l'ensemble courant, jusqu'à
    // ce qu'aucun ajout ne soit plus possible. Le résultat est unique (théorème de Dung).
    public static SortedSet<string> Grounded(AF af)
    {
        var current = new SortedSet<string>(StringComparer.Ordinal);
        bool changed = true;
        while (changed)
        {
            changed = false;
            foreach (var x in af.Args)
            {
                if (!current.Contains(x) && DefenseChecker.Defends(af, current, x))
                {
                    current.Add(x);
                    changed = true;
                }
            }
        }

        return current;
    }

    // Extensions preferred : ensembles admissibles maximaux par inclusion.
    public static List<SortedSet<string>> Preferred(AF af, IEnumerable<string> order = null)
    {
        var args = (order ?? af.Args).ToArray();
        var admissible = Powerset(args)
            .Select(combo => new SortedSet<string>(combo, StringComparer.Ordinal))
            .Where(set => AdmissibilityChecker.IsAdmissible(af, set))
            .ToList();

        // maximal : aucun autre admissible ne le contient strictement
        return admissible
            .Where(candidate => !admissible.Any(other => candidate.IsProperSubsetOf(other)))
            .ToList();
    }

    // Extensions stables : admissibles qui défont tout argument hors de l'extension.
    public static List<SortedSet<string>> Stable(AF af, IEnumerable<string> order = null)
    {
        var args = (order ?? af.Args).ToArray();
        var result = new List<SortedSet<string>>();
        foreach (var combo in Powerset(args))
        {
            var set = new SortedSet<string>(combo, StringComparer.Ordinal);
            if (AdmissibilityChecker.IsAdmissible(af, set)
                && af.Args.All(x => set.Contains(x) || DefenseChecker.Defeats(af, set, x)))
            {
                result.Add(set);
            }
        }

        return result;
    }

    // Affichage compact d'un ensemble d'arguments : {a,d}.
    public static string Fmt(IEnumerable<string> set)
    {
        return "{" + string.Join(",", set.OrderBy(x => x, StringComparer.Ordinal)) + "}";
    }

    // Liste d'extensions : ['{b,d}', '{a,d}'].
    public static string FmtList(IEnumerable<SortedSet<string>> sets)
    {
        return "[" + string.Join(", ", sets.Select(set => "'" + Fmt(set) + "'")) + "]";
    }

    // Repr du cadre : AF(args=['a', 'b', 'c', 'd'], attacks=[a->b, b->a, c->c]).
    public static string AfRepr(AF af)
    {
        var args = string.Join(
            ", ",
            af.Args.OrderBy(a => a, StringComparer.Ordinal).Select(a => "'" + a + "'"));
        var attacks = string.Join(
            ", ",
            af.Attacks
                .OrderBy(attack => attack.Attacker, StringComparer.Ordinal)
                .ThenBy(attack => attack.Target, StringComparer.Ordinal)
                .Select(attack => $"{attack.Attacker}->{attack.Target}"));

        return $"AF(args=[{args}], attacks=[{attacks}])";
    }

    // Comparaison setwise structurelle : chaque extension est comparée comme un ensemble
    // de symboles, sans passer par une représentation textuelle susceptible de confondre
    // l'argument "a,b" avec le couple d'arguments "a" et "b".
    public static bool SetwiseEquals(
        IEnumerable<SortedSet<string>> actual,
        IEnumerable<string[]> expected)
    {
        var actualSets = actual
            .Select(set => set.ToHashSet(StringComparer.Ordinal))
            .ToList();
        var expectedSets = expected
            .Select(set => set.ToHashSet(StringComparer.Ordinal))
            .ToList();

        return actualSets.Count == expectedSets.Count
            && actualSets.All(actualSet =>
                expectedSets.Count(expectedSet => actualSet.SetEquals(expectedSet)) == 1)
            && expectedSets.All(expectedSet =>
                actualSets.Count(actualSet => actualSet.SetEquals(expectedSet)) == 1);
    }

    // Calcule et affiche les trois sémantiques d'un AF (rend le triplet calculé).
    public static (SortedSet<string>, List<SortedSet<string>>, List<SortedSet<string>>) Resume(
        AF af,
        string label,
        IEnumerable<string> order = null)
    {
        var grounded = Grounded(af);
        var preferred = Preferred(af, order);
        var stable = Stable(af, order);

        Console.WriteLine($"=== {label} ===");
        var groundedText = grounded.Count == 0
            ? "∅"
            : string.Join(", ", grounded);
        Console.WriteLine($"  grounded  : {{{groundedText}}}   (unique, sceptique)");
        Console.WriteLine($"  preferred : {FmtList(preferred)}   ({preferred.Count} ext., crédule)");
        if (stable.Count == 0)
        {
            Console.WriteLine("  stable    : AUCUNE (pas d'extension stable)");
        }
        else
        {
            Console.WriteLine($"  stable    : {FmtList(stable)}   ({stable.Count} ext.)");
        }

        Console.WriteLine();
        return (grounded, preferred, stable);
    }
}

## 4. Le cas canonique : grounded $\neq$ preferred $\neq$ stable

Pour que l'écart entre les trois sémantiques soit **visible**, il faut un AF assez riche
pour que les trois attitudes divergent. Considérons :

$$F = \langle \{a, b, c, d\}, \; \{(a,b), (b,a), (c,c)\} \rangle$$

- **$a$ et $b$ s'attaquent mutuellement** : ni l'un ni l'autre n'est attaqué par un tiers,
  aucun n'est défendu « gratuitement ». Ce sont des **arguments flottants** — on peut en
  accepter un (si l'on prend parti) mais pas les deux (ils se contredisent).
- **$c$ s'attaque lui-même** : un argument auto-réfuté ne peut *jamais* appartenir à un
  ensemble sans conflit (il violerait la cohérence minimale).
- **$d$ est isolé** : non attaqué, il sera accepté partout (point de référence stable).

Cet AF est **non trivial** précisément parce qu'aucune sémantique ne se réduit à une
évidence : on ne peut pas « prendre tous les non-attaqués » (ça donnerait $\{d\}$ en
ignorant les flottements) ni « tout ce qui se défend » (ça mélangerait $a$ et $b$).

In [6]:
// AF canonique : flottement a/b + auto-attaque c + isolé d
var af1 = new AF(
    new[] { "a", "b", "c", "d" },
    new[] { ("a", "b"), ("b", "a"), ("c", "c") });

// Ordre d'énumération des arguments pour le powerset : il fixe l'ordre de rendu des
// listes d'extensions ('{b,d}' avant '{a,d}'), comme dans l'exécution de référence.
var af1Order = new[] { "c", "b", "a", "d" };

Console.WriteLine(Semantics.AfRepr(af1));
Console.WriteLine();

var (g1, p1, s1) = Semantics.Resume(
    af1,
    "F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}>",
    af1Order);

AF(args=['a', 'b', 'c', 'd'], attacks=[a->b, b->a, c->c])


=== F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}> ===


  grounded  : {d}   (unique, sceptique)


  preferred : ['{b,d}', '{a,d}']   (2 ext., crédule)


  stable    : AUCUNE (pas d'extension stable)


### Lecture du verdict

Les trois sémantiques donnent des résultats **différents** sur ce graphe :

| Sémantique | Résultat | Interprétation |
|------------|----------|----------------|
| **Grounded** | $\{d\}$ | Seul $d$ est *certainement* acceptable : il est le seul non attaqué. On refuse de trancher entre $a$ et $b$ (aucun n'est défendu sans prendre parti), et $c$ s'exclut de lui-même. Attitude sceptique : on n'accepte que l'incontestable. |
| **Preferred** | $\{a, d\}$ **et** $\{b, d\}$ | Deux positions maximales défendables, **incompatibles** : soit on accepte $a$ (qui défait $b$), soit $b$ (qui défait $a$). Attitude crédule : on admet chaque position auto-défendable maximale, même s'il en existe une contradictoire. |
| **Stable** | *(aucune)* | Aucune extension stable n'existe. Pourquoi ? Une extension stable devrait défaire *tout* argument extérieur. Mais $c$ n'est attaqué par personne d'autre que lui-même : aucun argument ne peut le défaire « de l'extérieur », donc aucune extension stable ne peut le statuer. Le graphe porte un argument « indécidable de l'extérieur » → la sémantique stable échoue à exister. |

C'est précisément le théorème de Dung rendu visible :
- **grounded $\subset$ preferred** ($\{d\} \subset \{a,d\}$) ;
- **preferred $\ne$ stable** (preferred existe, stable n'existe pas) ;
- la sémantique **stable n'est pas garantie d'existence** — c'est sa fragilité théorique,
  et la raison pour laquelle les solveurs pratiques (Tweety inclus) calculent en priorité
  grounded et preferred.

In [7]:
// Vérification programmatique : les trois sémantiques divergent bien.

var expectedGrounded = new[] { "d" };
var expectedPreferred = new[] { new[] { "a", "d" }, new[] { "b", "d" } };

var expectedGroundedText = "{" + string.Join(",", expectedGrounded) + "}";
var expectedPreferredText =
    "{{" + string.Join(",", expectedPreferred[0]) + "},{" + string.Join(",", expectedPreferred[1]) + "}}";

var groundedOk = g1.SetEquals(expectedGrounded);
var preferredOk = Semantics.SetwiseEquals(p1, expectedPreferred);
var stableAbsent = s1.Count == 0;
var stableExists = !stableAbsent;

Console.WriteLine($"grounded == {expectedGroundedText} ? {groundedOk}");
Console.WriteLine(
    $"  attendu {expectedGroundedText} vs calculé {Semantics.Fmt(g1)} : {(groundedOk ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"preferred == {expectedPreferredText} ? {preferredOk}");
Console.WriteLine(
    $"  attendu {expectedPreferredText} vs calculé {Semantics.FmtList(p1)} : {(preferredOk ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"stable existe ? {stableExists}");
Console.WriteLine(
    $"  attendu False vs calculé {stableExists} : {(stableAbsent ? "CONFORME" : "ÉCART")}");
Console.WriteLine();

// La grounded est strictement incluse dans la première extension preferred.
var strictSubset = p1.Count > 0 && g1.IsProperSubsetOf(p1[0]);
Console.WriteLine($">>> grounded ⊊ preferred : {strictSubset}");
Console.WriteLine($"  attendu Vrai vs calculé {strictSubset} : {(strictSubset ? "CONFORME" : "ÉCART")}");

var divergence = p1.Count > 0 && s1.Count == 0;
Console.WriteLine($">>> preferred existe ET stable n'existe pas (divergence) : {divergence}");
Console.WriteLine($"  attendu Vrai vs calculé {divergence} : {(divergence ? "CONFORME" : "ÉCART")}");

// Assertions exécutables : l'exécution échoue si une parité calculée est rompue.
if (!groundedOk)
{
    throw new InvalidOperationException("La grounded calculée ne correspond pas à la source.");
}
if (!preferredOk)
{
    throw new InvalidOperationException("Les extensions preferred calculées ne correspondent pas à la source.");
}
if (!stableAbsent)
{
    throw new InvalidOperationException("La stable ne devrait pas exister sur l'AF canonique.");
}
if (!strictSubset)
{
    throw new InvalidOperationException("La grounded doit être strictement incluse dans preferred.");
}
if (!divergence)
{
    throw new InvalidOperationException("Preferred doit exister et stable ne doit pas exister sur l'AF canonique.");
}

grounded == {d} ? True


  attendu {d} vs calculé {d} : CONFORME


preferred == {{a,d},{b,d}} ? True


  attendu {{a,d},{b,d}} vs calculé ['{b,d}', '{a,d}'] : CONFORME


stable existe ? False


  attendu False vs calculé False : CONFORME


>>> grounded ⊊ preferred : True


  attendu Vrai vs calculé True : CONFORME


>>> preferred existe ET stable n'existe pas (divergence) : True


  attendu Vrai vs calculé True : CONFORME


## 4bis. Quand stable n'existe pas : semi-stable et idéale

Le verdict de la section 4 laisse un problème ouvert : sur `af1`, **la sémantique stable n'existe
pas** — $c$, l'auto-attaqué, ne peut être statué par aucune extension. C'est la « fragilité »
relevée plus haut, et le curriculum la rencontrera à nouveau : [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb)
la constatera sur un cycle impair, où il introduira CF2 — une réponse qui **abandonne
l'admissibilité**. Or la littérature dispose d'une réparation qui reste dans le cadre : deux
sémantiques, absentes de ce notebook jusqu'ici.

- **Semi-stable** (Caminada 2006) : parmi les ensembles **admissibles**, ceux qui maximisent la
  *portée* $S \cup S^+$ — les arguments **acceptés** ou **attaqués** par $S$. Statuer sur le
  maximum : c'est la stable « autant que possible ». Trois propriétés la désignent comme la
  réparation canonique :
  1. elle **existe toujours** pour un AF fini — le défaut constaté sur `af1` est réparé par construction ;
  2. elle **coïncide avec la stable** dès qu'une extension stable existe — rien n'est perdu ;
  3. elle reste **admissible** — elle ne quitte pas le cadre construit en section 2.

- **Idéale** (Dung, Mancarella & Toni 2007) : la plus grande extension admissible contenue dans
  **toutes** les extensions preferred. C'est le point manquant de l'axe sceptique/crédule du
  tableau de la section 3 : un verdict **unique** — comme la grounded — mais **moins timide**,
  puisqu'il inclut tout ce qui est défendable dans *chacune* des positions crédules. Elle
  vérifie l'encadrement $\text{grounded} \subseteq \text{idéale} \subseteq \bigcap \text{preferred}$.

Implémentons les deux par-dessus la machinerie des sections 2-3 (`powerset`, `is_admissible`,
`preferred`) — quelques lignes chacune, aucune dépendance nouvelle.

In [8]:
// Illustration 6 — Semi-stable : la réparation de la stable.

// Retour sur l'AF canonique af1 — là où la sémantique stable n'existe pas.
var af1Semi = new AF(
    new[] { "a", "b", "c", "d" },
    new[] { ("a", "b"), ("b", "a"), ("c", "c") });
var af1SemiOrder = new[] { "c", "b", "a", "d" };

var s1Semi = Semantics.Stable(af1Semi, af1SemiOrder);
var p1Semi = Semantics.Preferred(af1Semi, af1SemiOrder);
var ss1 = SemiStableSemantics.SemiStable(af1Semi, af1SemiOrder);

Console.WriteLine("=== af1 : le défaut et sa réparation ===");
Console.WriteLine(
    $"  stable      : {(s1Semi.Count == 0 ? "AUCUNE" : Semantics.FmtList(s1Semi))}");
Console.WriteLine($"  semi-stable : {Semantics.FmtList(ss1)}   ({ss1.Count} ext.)");
Console.WriteLine();

var repairs = ss1.Count > 0 && s1Semi.Count == 0;
Console.WriteLine($"semi-stable existe là où stable n'existe pas : {repairs}");
Console.WriteLine($"  attendu Vrai vs calculé {repairs} : {(repairs ? "CONFORME" : "ÉCART")}");

var sameAsPreferred = Semantics.SetwiseEquals(ss1, p1Semi.Select(set => set.ToArray()));
Console.WriteLine($"sur af1, semi-stable == preferred            : {sameAsPreferred}");
Console.WriteLine($"  attendu Vrai vs calculé {sameAsPreferred} : {(sameAsPreferred ? "CONFORME" : "ÉCART")}");

if (!repairs)
{
    throw new InvalidOperationException("La semi-stable doit réparer l'absence de stable sur af1.");
}
if (!sameAsPreferred)
{
    throw new InvalidOperationException("La semi-stable doit coïncider avec preferred sur af1.");
}

public static class SemiStableSemantics
{
    // S ∪ S+ : la portée de S — les arguments acceptés par S ou attaqués par S.
    public static SortedSet<string> RangeOf(AF af, IEnumerable<string> set)
    {
        var members = set.ToHashSet(StringComparer.Ordinal);
        var range = new SortedSet<string>(members, StringComparer.Ordinal);
        foreach (var attack in af.Attacks)
        {
            // L'appartenance se teste sur S (et non sur la portée en croissance).
            if (members.Contains(attack.Attacker))
            {
                range.Add(attack.Target);
            }
        }

        return range;
    }

    // Extensions semi-stables : les admissibles qui maximisent |S ∪ S+| (Caminada 2006).
    public static List<SortedSet<string>> SemiStable(AF af, IEnumerable<string> order = null)
    {
        var args = (order ?? af.Args).ToArray();
        var admissible = Semantics.Powerset(args)
            .Select(combo => new SortedSet<string>(combo, StringComparer.Ordinal))
            .Where(set => AdmissibilityChecker.IsAdmissible(af, set))
            .ToList();

        var best = admissible.Max(set => RangeOf(af, set).Count);
        return admissible.Where(set => RangeOf(af, set).Count == best).ToList();
    }
}

=== af1 : le défaut et sa réparation ===


  stable      : AUCUNE


  semi-stable : ['{b,d}', '{a,d}']   (2 ext.)


semi-stable existe là où stable n'existe pas : True


  attendu Vrai vs calculé True : CONFORME


sur af1, semi-stable == preferred            : True


  attendu Vrai vs calculé True : CONFORME


In [9]:
// Propriété 2 : quand la stable existe, la semi-stable coïncide avec elle.
// af3 : un AF acyclique (DAG d'attaque) — la stable y existe toujours.
var af3 = new AF(
    new[] { "a", "b", "c" },
    new[] { ("a", "b") });
var af3Order = new[] { "a", "b", "c" };

var s3 = Semantics.Stable(af3, af3Order);
var ss3 = SemiStableSemantics.SemiStable(af3, af3Order);
var p3 = Semantics.Preferred(af3, af3Order);

Console.WriteLine("=== af3 = AF(args={a,b,c}, attacks={a->b}) — la stable existe ===");
Console.WriteLine($"  stable      : {Semantics.FmtList(s3)}");
Console.WriteLine($"  semi-stable : {Semantics.FmtList(ss3)}");
Console.WriteLine($"  preferred   : {Semantics.FmtList(p3)}");
Console.WriteLine();

// Parité des valeurs avec la source Python : {a,c} pour les trois sémantiques.
var attenduAf3 = new[] { new[] { "a", "c" } };
var attenduAf3Text = "{" + string.Join(",", attenduAf3[0]) + "}";
var stableOk = s3.Count > 0 && Semantics.SetwiseEquals(s3, attenduAf3);
var semiStableOk = ss3.Count > 0 && Semantics.SetwiseEquals(ss3, attenduAf3);
var preferredOk = p3.Count > 0 && Semantics.SetwiseEquals(p3, attenduAf3);
Console.WriteLine($"  stable      == {attenduAf3Text} : {stableOk}  -> {(stableOk ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"  semi-stable == {attenduAf3Text} : {semiStableOk}  -> {(semiStableOk ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"  preferred   == {attenduAf3Text} : {preferredOk}  -> {(preferredOk ? "CONFORME" : "ÉCART")}");

var coincide = Semantics.SetwiseEquals(ss3, s3.Select(set => set.ToArray()));
Console.WriteLine($"coïncidence semi-stable == stable (setwise) : {coincide}");
Console.WriteLine($"  attendu Vrai vs calculé {coincide} : {(coincide ? "CONFORME" : "ÉCART")}");

if (!stableOk)
{
    throw new InvalidOperationException("La stable de af3 doit être exactement {a,c}.");
}
if (!semiStableOk)
{
    throw new InvalidOperationException("La semi-stable de af3 doit être exactement {a,c}.");
}
if (!preferredOk)
{
    throw new InvalidOperationException("La preferred de af3 doit être exactement {a,c}.");
}
if (!coincide)
{
    throw new InvalidOperationException("La semi-stable doit coïncider avec la stable sur af3.");
}

=== af3 = AF(args={a,b,c}, attacks={a->b}) — la stable existe ===


  stable      : ['{a,c}']


  semi-stable : ['{a,c}']


  preferred   : ['{a,c}']


  stable      == {a,c} : True  -> CONFORME


  semi-stable == {a,c} : True  -> CONFORME


  preferred   == {a,c} : True  -> CONFORME


coïncidence semi-stable == stable (setwise) : True


  attendu Vrai vs calculé True : CONFORME


### Lecture du verdict : la réparation en action

Là où la stable affichait « AUCUNE », la semi-stable rend $\{a,d\}$ et $\{b,d\}$ — exactement
les preferred de `af1`. Ce n'est pas un hasard : une extension qui statue sur un maximum
d'arguments est nécessairement crédule-maximale. La différence entre les deux notions se joue
sur $c$ : personne ne l'attaque « de l'extérieur » (il ne s'attaque que lui-même, et cela ne
compte pas pour la portée d'une autre extension), donc **aucune** portée ne couvre $c$ — la
semi-stable ne décide pas $c$, elle décide **tout le reste** et laisse $c$ explicitement
indécidable. C'est le compromis exact : maximiser les arguments statués **sans jamais
sacrifier l'admissibilité** — là où CF2 (Tweety-5) paiera l'existence en quittant le cadre
admissible.

*(On montre — c'est l'esprit de l'Exercice 1 — que ces ensembles de portée maximale sont
automatiquement complets.)*

### La sémantique idéale : un verdict unique sur l'axe sceptique

La preferred est crédule : elle rend **plusieurs** verdicts incompatibles. La grounded est
sceptique : elle n'ose que l'incontestable — parfois $\emptyset$, comme sur les témoins
ci-dessous. Entre les deux, la sémantique **idéale** répond à la question que pose le pipeline
agentique de la série : *quelle position unique peut tenir un système qui doit trancher sans
prendre parti entre les extensions preferred ?* La réponse : tout ce qui est défendable **dans
chacune** des positions crédules — l'admissible maximale contenue dans toutes les preferred.
Elle existe toujours, est unique, et l'encadrement
$\text{grounded} \subseteq \text{idéale} \subseteq \bigcap\text{preferred}$ se vérifie
programmatiquement — sur deux témoins où l'écart est visible de part et d'autre.

In [10]:
// Illustration 7 — La sémantique idéale : un verdict unique sur l'axe sceptique.

// Témoin 1 : a et b flottent ; x et y se défendent mutuellement contre eux.
var afId1 = new AF(
    new[] { "a", "b", "x", "y" },
    new[] { ("a", "b"), ("b", "a"), ("a", "x"), ("y", "a"), ("b", "y"), ("x", "b") });
var afId1Order = new[] { "b", "a", "y", "x" };

// Témoin 2 : a et b flottent ; w (auto-attaqué) menace z, que seuls a et b savent défaire.
var afId2 = new AF(
    new[] { "a", "b", "w", "z" },
    new[] { ("a", "b"), ("b", "a"), ("w", "w"), ("w", "z"), ("a", "w"), ("b", "w") });
var afId2Order = new[] { "b", "a", "w", "z" };

// Valeurs de référence du notebook Python (parité attendue).
var references = new (string Name, string[][] Preferred, string[] Grounded, string[] Ideal, string[] Intersection)[]
{
    ("af_id1", new[] { new[] { "x", "y" } }, Array.Empty<string>(), new[] { "x", "y" }, new[] { "x", "y" }),
    ("af_id2", new[] { new[] { "a", "z" }, new[] { "b", "z" } }, Array.Empty<string>(), Array.Empty<string>(), new[] { "z" }),
};

foreach (var witness in new[]
{
    ("af_id1", afId1, afId1Order),
    ("af_id2", afId2, afId2Order),
})
{
    var (name, afi, order) = witness;
    var reference = references.First(entry => entry.Name == name);

    var prefs = Semantics.Preferred(afi, order);
    var grounded = Semantics.Grounded(afi);
    var ideal = IdealSemantics.Ideal(afi, order);
    var intersection = IdealSemantics.Intersection(prefs);

    Console.WriteLine($"=== {name} ===");
    Console.WriteLine($"  preferred   : {Semantics.FmtList(prefs)}");
    Console.WriteLine($"  grounded    : {Semantics.Fmt(grounded)}");
    Console.WriteLine($"  idéale      : {Semantics.Fmt(ideal)}");
    Console.WriteLine($"  ∩ preferred : {Semantics.Fmt(intersection)}");

    // Parité des valeurs affichées avec la source Python.
    var preferredOk = prefs.Count > 0 && Semantics.SetwiseEquals(prefs, reference.Preferred);
    var groundedOk = grounded.SetEquals(reference.Grounded);
    var idealOk = ideal.SetEquals(reference.Ideal);
    var intersectionOk = intersection.SetEquals(reference.Intersection);
    var valuesOk = preferredOk && groundedOk && idealOk && intersectionOk;
    Console.WriteLine($"  valeurs identiques à la source (preferred, grounded, idéale, ∩preferred) : {valuesOk}");
    Console.WriteLine($"  attendu Vrai vs calculé {valuesOk} : {(valuesOk ? "CONFORME" : "ÉCART")}");

    // Encadrement g ⊆ i ⊆ ∩pref : recalculé, jamais supposé.
    var framed = grounded.IsSubsetOf(ideal) && ideal.IsSubsetOf(intersection);
    Console.WriteLine($"  encadrement g ⊆ i ⊆ ∩pref : {framed}");
    Console.WriteLine($"  attendu Vrai vs calculé {framed} : {(framed ? "CONFORME" : "ÉCART")}");

    // Contrôle de la source : les trois ensembles ne sont pas tous égaux.
    var allEqual = grounded.SetEquals(ideal) && ideal.SetEquals(intersection);
    var notAllEqual = !allEqual;
    Console.WriteLine($"  les trois ensembles ne sont pas tous égaux : {notAllEqual}");
    Console.WriteLine($"  attendu Vrai vs calculé {notAllEqual} : {(notAllEqual ? "CONFORME" : "ÉCART")}");

    // Complément : les trois sont-ils distincts deux à deux ?
    var allDistinct = !grounded.SetEquals(ideal)
        && !grounded.SetEquals(intersection)
        && !ideal.SetEquals(intersection);
    Console.WriteLine($"  (complément) les trois ensembles sont distincts deux à deux : {allDistinct}");
    Console.WriteLine($"  attendu Faux vs calculé {allDistinct} : {(allDistinct ? "ÉCART" : "CONFORME")}");

    Console.WriteLine();

    if (!preferredOk)
    {
        throw new InvalidOperationException($"Les extensions preferred de {name} ne correspondent pas à la source.");
    }
    if (!groundedOk)
    {
        throw new InvalidOperationException($"La grounded de {name} ne correspond pas à la source.");
    }
    if (!idealOk)
    {
        throw new InvalidOperationException($"L'idéale de {name} ne correspond pas à la source.");
    }
    if (!intersectionOk)
    {
        throw new InvalidOperationException($"L'intersection des preferred de {name} ne correspond pas à la source.");
    }
    if (!framed)
    {
        throw new InvalidOperationException($"L'encadrement ground ⊆ idéal ⊆ intersection preferred échoue pour {name}.");
    }
    if (!notAllEqual)
    {
        throw new InvalidOperationException($"Les trois ensembles de {name} ne devraient pas être tous égaux.");
    }
}

public static class IdealSemantics
{
    // ∩ preferred : les arguments présents dans TOUTES les extensions preferred.
    public static SortedSet<string> Intersection(IReadOnlyList<SortedSet<string>> preferred)
    {
        var intersection = new SortedSet<string>(StringComparer.Ordinal);
        if (preferred.Count == 0)
        {
            return intersection;
        }

        intersection = new SortedSet<string>(preferred[0], StringComparer.Ordinal);
        foreach (var extension in preferred.Skip(1))
        {
            intersection.IntersectWith(extension);
        }

        return intersection;
    }

    // L'extension idéale : la plus grande admissible contenue dans TOUTES les preferred.
    // (Dung, Mancarella & Toni 2007.) Toujours existante et unique : l'union d'admissibles
    // deux à deux compatibles est admissible, donc le plus grand candidat est l'unique maximal.
    public static SortedSet<string> Ideal(AF af, IEnumerable<string> order = null)
    {
        var preferred = Semantics.Preferred(af, order);
        var intersection = Intersection(preferred);
        var candidates = Semantics.Powerset(intersection)
            .Select(combo => new SortedSet<string>(combo, StringComparer.Ordinal))
            .Where(set => AdmissibilityChecker.IsAdmissible(af, set))
            .ToList();

        return candidates
            .OrderByDescending(set => set.Count)
            .ThenBy(set => string.Join(",", set), StringComparer.Ordinal)
            .First();
    }
}

=== af_id1 ===


  preferred   : ['{x,y}']


  grounded    : {}


  idéale      : {x,y}


  ∩ preferred : {x,y}


  valeurs identiques à la source (preferred, grounded, idéale, ∩preferred) : True


... (16 outputs left out)

  les trois ensembles ne sont pas tous égaux : True


  attendu Vrai vs calculé True : CONFORME


  (complément) les trois ensembles sont distincts deux à deux : False


  attendu Faux vs calculé False : CONFORME


### Lecture des deux témoins : le sandwich rendu visible

Sur `af_id1`, $x$ et $y$ se défendent **mutuellement** ($y$ défait l'attaquant de $x$, $x$
défait celui de $y$) : aucun n'est défendu à partir de $\emptyset$, donc la grounded reste
vide — mais ensemble ils sont admissibles, et présents dans l'unique extension preferred.
L'idéale accepte alors $\{x,y\}$ là où la grounded ne disait rien : **moins timide**, pour
le même prix — un verdict unique.

Sur `af_id2`, l'argument $z$ est présent dans **chaque** preferred — que l'on choisisse $a$ ou
$b$ comme parti pris, l'un des deux défait $w$, l'agresseur de $z$. Donc
$z \in \bigcap\text{preferred}$. Mais cette intersection n'est **pas** admissible : $z$ n'y
est défendu par personne ($w$ n'est défait que par $a$ ou par $b$, jamais ensemble dans une
même extension). L'idéale retombe alors sur $\emptyset$ : elle ne promet que ce qui est
défendable **sans choisir** de camp.

Les deux témoins ensemble situent exactement la sémantique : strictement au-dessus de la
grounded dès qu'existe une défense mutuelle universelle, strictement sous l'intersection dès
qu'un argument commun n'est défendu que par des partis pris. Construire un AF où les **trois**
diffèrent deux à deux est l'objet de l'Exercice 4.

## 5. Retour au solveur Tweety

Maintenant que les sémantiques sont transparentes, relisons l'appel du notebook
[2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) :

```python
reasoner_d = SimpleGroundedReasoner()
models = reasoner_d.getModels(theory)   # Collection<Extension>
```

`getModels` renvoie la collection des extensions selon la sémantique du *reasoner*. Pour
`SimpleGroundedReasoner`, c'est exactement le point fixe calculé par notre fonction
`grounded` ci-dessus — une seule extension, la plus prudente. Les *reasoners*
`SimplePreferredReasoner` et `SimpleCompleteReasoner` implémentent les autres colonnes
du tableau. Le pont JPype ne fait rien d'autre qu'appeler, en JVM, l'algorithme que nous
venons de reconstruire à la main.

**Ce que ce notebook apporte** au-delà de Tweety-5 : la compréhension de l'algorithme,
pas seulement son usage. C'est la différence entre *invoquer* un vérificateur formel
(le pipeline agentique de la série) et *comprendre* ce qu'il calcule (ce notebook).

> Pour les preuves formelles des théorèmes d'inclusion et d'existence (grounded est
> l'extension complète minimale, stable peut être vide), voir le notebook Lean
> [Tweety-5b](../Tweety/Tweety-05b-Argumentation-Lean.ipynb).

## 6. Exercices

Les cinq exercices suivants approfondissent la théorie. Ils sont laissés **incomplets**
par construction (stub) : à vous de les remplir. Le notebook s'exécute de bout en bout
même non complété — vos implémentations remplaceront les `// TODO etudiant`.


### Exercice 1 — Complétude vs admissibilité

Une extension **complète** est admissible *et* contient tout argument qu'elle défend :
$S$ complète $\iff$ $S$ admissible et $\forall x \in A$, si $S$ défend $x$ alors $x \in S$.
La grounded est l'unique complète minimale, les preferred sont les complètes maximales.

**Objectif** : implémenter `IsComplete(af, S)` et vérifier que la grounded et les
preferred de `af1` sont bien complètes (mais qu'un admissible non-maximal comme $\{a\}$
ne l'est pas — pourquoi ?).

*Indice* : un argument défendu mais absent de $S$ suffit à le disqualifier.

In [11]:
// Exercice 1 : compléter cette fonction
public static class CompletenessStub
{
    // True si S est complet : admissible ET contient tout argument qu'il défend.
    // Etape 1 : vérifier l'admissibilité (déjà disponible : IsAdmissible).
    // Etape 2 : pour tout x défendu par S, x doit appartenir à S.
    // TODO etudiant
    public static bool IsComplete(AF af, IEnumerable<string> candidates)
    {
        return false;
    }
}

// Test (à décommenter une fois complété) :
// Console.WriteLine($"grounded complet ? {CompletenessStub.IsComplete(af1, g1)}");
// Console.WriteLine($"preferred complet ? {p1.All(extension => CompletenessStub.IsComplete(af1, extension))}");
// Console.WriteLine($"{{a}} (admissible non-maximal) complet ? {CompletenessStub.IsComplete(af1, new[] { "a" })}");

### Exercice 2 — Un AF où la sémantique stable existe

Sur `af1`, la sémantique stable n'existait pas (à cause de l'auto-attaque de $c$).
Construisez un AF **sans auto-attaque ni cycle impair** où la stable existe *et* coïncide
avec la preferred.

**Objectif** : définir `af2` et vérifier que `Stable(af2) == Preferred(af2)` (même
collection d'extensions).

*Indice* : un AF acyclique (les arguments forment un DAG d'attaque) admet toujours une
unique extension stable, qui coïncide avec la grounded et la preferred.

In [12]:
// Exercice 2 : définir un AF où stable == preferred
var af2 = new AF(
    Array.Empty<string>(),                 // TODO etudiant : choisir args
    Array.Empty<(string, string)>());      // TODO etudiant : choisir attacks (sans auto-attaque)

// Test (à décommenter une fois complété) :
// var (g2, p2, s2) = Semantics.Resume(af2, "af2 (votre AF)");
// Console.WriteLine($"stable == preferred ? {Semantics.SetwiseEquals(s2, p2.Select(extension => extension.ToArray()))}");

### Exercice 3 — Étiquetage IN / OUT / UNDEC

Les sémantiques admettent une formulation équivalente en **étiquetages** : chaque
argument reçoit l'étiquette IN (accepté), OUT (rejeté) ou UNDEC (indécidable), avec les
règles : (i) $x$ est OUT si un IN l'attaque ; (ii) $x$ est IN si tous ses attaquants sont
OUT. Le labeling grounded de `af1` devrait donner $d$=IN, $c$=UNDEC (auto-attaqué, mais rejeté par aucun IN), et
$a, b$=UNDEC (le flottement non tranché).

**Objectif** : implémenter `GroundedLabeling(af)` renvoyant un dictionnaire
`{arg: "IN"/"OUT"/"UNDEC"}`, et vérifier qu'il est cohérent avec `Grounded(af)` (IN $=$ l'extension
grounded, du moins quand aucun argument n'est UNDEC dans la grounded).

*Indice* : OUT = attaqué par un IN ; IN = non attaqué, ou tous ses attaquants OUT ;
UNDEC = ni IN ni OUT (le reste). Itérez jusqu'à stabilisation, comme pour le point fixe.

In [13]:
// Exercice 3 : compléter le labeling grounded
public static class GroundedLabelingStub
{
    // Etape 1 : IN = argument dont tous les attaquants sont OUT (ou sans attaquant).
    // Etape 2 : OUT = argument attaqué par au moins un IN.
    // Etape 3 : itérer jusqu'au point fixe (comme Grounded).
    // TODO etudiant
    public static Dictionary<string, string> GroundedLabeling(AF af)
    {
        var labels = new Dictionary<string, string>(StringComparer.Ordinal);
        foreach (var argument in af.Args)
        {
            labels[argument] = "UNDEC";
        }

        return labels;
    }
}

// Test (à décommenter une fois complété) :
// var lab = GroundedLabelingStub.GroundedLabeling(af1);
// Console.WriteLine($"Labeling grounded de af1 : {string.Join(", ", lab.Select(entry => $"{entry.Key}={entry.Value}"))}");
// Console.WriteLine($"Cohérent avec grounded={{d}} ? {lab.Where(entry => entry.Value == "IN").Select(entry => entry.Key).ToHashSet(StringComparer.Ordinal).SetEquals(g1)}");

### Exercice 4 — L'axe sceptique complet : grounded, idéale, $\bigcap$preferred

Le § 4bis a montré deux témoins : sur `af_id1`, l'idéale dépasse la grounded
($\emptyset \subsetneq \{x,y\}$) mais égale l'intersection ; sur `af_id2`, elle égale la
grounded ($\emptyset$) mais reste sous l'intersection ($\{z\}$). Dans chacun, **deux des trois
coïncident**.

**Objectif** : construire un AF où les trois ensembles diffèrent **deux à deux** —
$\text{grounded} \subsetneq \text{idéale} \subsetneq \bigcap\text{preferred}$, strictement.

**Indices :**
- *Indice 1* : pour dépasser la grounded, il faut de la **défense mutuelle** — des arguments
  défendus seulement par d'autres arguments eux-mêmes non défendus à partir de $\emptyset$,
  mais présents dans toutes les preferred.
- *Indice 2* : pour rester sous l'intersection, il faut un argument **commun à toutes les
  preferred** mais défendu uniquement par des arguments qui changent selon l'extension.
- *Etape 1* : écrire `af4` (les deux mécanismes des témoins peuvent coexister dans un même
  AF) ; *Etape 2* : décommenter le test ; *Etape 3* : vérifier que les trois ensembles
  forment bien une chaîne strictement croissante.

In [14]:
// Exercice 4 : construire un AF où grounded, idéale et ∩preferred diffèrent DEUX À DEUX
var af4 = new AF(
    Array.Empty<string>(),                 // TODO etudiant : choisir args
    Array.Empty<(string, string)>());      // TODO etudiant : choisir attacks

// Test (à décommenter une fois complété) :
// var g4 = Semantics.Grounded(af4);
// var i4 = IdealSemantics.Ideal(af4);
// var inter4 = IdealSemantics.Intersection(Semantics.Preferred(af4));
// Console.WriteLine($"grounded : {Semantics.Fmt(g4)} | idéale : {Semantics.Fmt(i4)} | ∩preferred : {Semantics.Fmt(inter4)}");
// var strictGroundedIdeal = g4.IsProperSubsetOf(i4);
// var strictIdealIntersection = i4.IsProperSubsetOf(inter4);
// Console.WriteLine($"grounded ⊊ idéale : {strictGroundedIdeal}");
// Console.WriteLine($"idéale ⊊ ∩preferred : {strictIdealIntersection}");
// if (!strictGroundedIdeal || !strictIdealIntersection)
// {
//     throw new InvalidOperationException("Les deux inclusions strictes attendues ne sont pas vérifiées.");
// }

Console.WriteLine("Exercice a completer : axe sceptique grounded / idéale / ∩preferred");

Exercice a completer : axe sceptique grounded / idéale / ∩preferred


### Aller plus loin — la trajectoire d’étiquetage

Jusqu’ici, le cadre AF et ses sémantiques ont été calculés sur un **graphe complet** : tous
les arguments et toutes les attaques sont présents dès le départ. Pour observer comment un
verdict se construit lorsque les informations arrivent progressivement, on peut reconstruire un
graphe **partiel** à chaque étape.

Dans cette démonstration, les arguments arrivent un par un. Après chaque arrivée, on ne
conserve que les arguments déjà livrés et les attaques dont **les deux extrémités** sont
présentes. On recalcule alors l’étiquetage grounded sur ce graphe partiel, en notant chaque
argument `IN`, `OUT` ou `UNDEC`.

Ce changement de point de vue est important : une attaque n’est pas encore active tant que
son auteur et sa cible ne sont pas tous deux arrivés. Un argument peut donc être accepté
provisoirement, puis révisé lorsque l information manquante arrive. La trajectoire conserve
ainsi l’ordre des révisions, information que le seul étiquetage final ne révèle pas.

La cellule suivante met en scène deux ordres d’arrivée sur l’AF canonique et compare leurs
étapes, avant de proposer l’exercice 5 à la fin de cette démonstration.

In [15]:
// Raccourci assumé : on lit les étiquettes depuis `Grounded()`, déjà fourni plus haut.
// ⚠ Ce n'est PAS la solution de l'Exercice 3, qui demande la formulation par point fixe
// (itération IN/OUT jusqu'à stabilisation) — celle-là reste à écrire.

var ordre1 = new[] { "d", "c", "a", "b" };
var ordre2 = new[] { "a", "b", "c", "d" };

var fin1 = TrajectoireDemo.Montre(af1, ordre1);
Console.WriteLine();
var fin2 = TrajectoireDemo.Montre(af1, ordre2);
Console.WriteLine();

var finalIdentique = TrajectoireDemo.EtiquettesEgales(fin1, fin2);
Console.WriteLine($"Étiquetage FINAL identique ? {finalIdentique}");

var etapes1 = TrajectoireDemo.Trajectoire(af1, ordre1);
var etapes2 = TrajectoireDemo.Trajectoire(af1, ordre2);
var trajectoiresIdentiques = TrajectoireDemo.EtapesEgales(etapes1, etapes2);
Console.WriteLine($"Trajectoires identiques    ? {trajectoiresIdentiques}");

// Contrôles exécutables : étiquettes attendues à chaque étape (parité avec la source).
var attendu1 = new[]
{
    new Dictionary<string, string> { ["d"] = "IN" },
    new Dictionary<string, string> { ["d"] = "IN", ["c"] = "UNDEC" },
    new Dictionary<string, string> { ["d"] = "IN", ["c"] = "UNDEC", ["a"] = "IN" },
    new Dictionary<string, string> { ["d"] = "IN", ["c"] = "UNDEC", ["a"] = "UNDEC", ["b"] = "UNDEC" },
};

var attendu2 = new[]
{
    new Dictionary<string, string> { ["a"] = "IN" },
    new Dictionary<string, string> { ["a"] = "UNDEC", ["b"] = "UNDEC" },
    new Dictionary<string, string> { ["a"] = "UNDEC", ["b"] = "UNDEC", ["c"] = "UNDEC" },
    new Dictionary<string, string> { ["a"] = "UNDEC", ["b"] = "UNDEC", ["c"] = "UNDEC", ["d"] = "IN" },
};

foreach (var (ordre, etapes, attendu, nom) in new[]
{
    (ordre1, etapes1, attendu1, "d->c->a->b"),
    (ordre2, etapes2, attendu2, "a->b->c->d"),
})
{
    for (int k = 0; k < attendu.Length; k++)
    {
        var ok = TrajectoireDemo.LabelsEgaux(attendu[k], etapes[k]);
        var attenduTexte = string.Join(", ", attendu[k].Select(entry => $"{entry.Key}={entry.Value}"));
        Console.WriteLine(
            $"  ordre {nom} — étape {k + 1} : attendu [{attenduTexte}] -> {(ok ? "CONFORME" : "ÉCART")}");
        if (!ok)
        {
            throw new InvalidOperationException($"Étiquetage d'étape inattendu (ordre {nom}, étape {k + 1}).");
        }
    }
}

if (!finalIdentique)
{
    throw new InvalidOperationException("Les étiquetages finaux doivent être indépendants de l'ordre d'arrivée.");
}
if (trajectoiresIdentiques)
{
    throw new InvalidOperationException("Les deux trajectoires de démonstration doivent différer.");
}

public static class TrajectoireDemo
{
    // {arg: 'IN'|'OUT'|'UNDEC'} dérivé de l'extension grounded de `af`.
    public static Dictionary<string, string> LabellingDepuisGrounded(AF af)
    {
        var extension = Semantics.Grounded(af);
        var labels = new Dictionary<string, string>(StringComparer.Ordinal);
        foreach (var x in af.Args)
        {
            if (extension.Contains(x))
            {
                labels[x] = "IN";
            }
            else if (af.Attacks.Any(attack => attack.Target == x && extension.Contains(attack.Attacker)))
            {
                labels[x] = "OUT";
            }
            else
            {
                labels[x] = "UNDEC";
            }
        }

        return labels;
    }

    // Étiquetage après chaque arrivée, sous un ordre de délivrance donné.
    // Une attaque ne devient ACTIVE que lorsque ses deux extrémités sont arrivées.
    public static List<Dictionary<string, string>> Trajectoire(AF af, IEnumerable<string> ordreArrivee)
    {
        var ordre = ordreArrivee.ToArray();
        if (!af.Args.SetEquals(ordre))
        {
            throw new ArgumentException("ordre_arrivee doit énumérer exactement af.args");
        }

        var etapes = new List<Dictionary<string, string>>();
        for (int k = 1; k <= ordre.Length; k++)
        {
            var arrives = ordre.Take(k).ToHashSet(StringComparer.Ordinal);
            var actives = af.Attacks
                .Where(attack => arrives.Contains(attack.Attacker) && arrives.Contains(attack.Target))
                .ToList();
            etapes.Add(LabellingDepuisGrounded(new AF(arrives, actives)));
        }

        return etapes;
    }

    public static Dictionary<string, string> Montre(AF af, IEnumerable<string> ordre)
    {
        var ordreArray = ordre.ToArray();
        Console.WriteLine($"Ordre d'arrivée : {string.Join(" -> ", ordreArray)}");
        var etapes = Trajectoire(af, ordreArray);
        for (int k = 1; k <= etapes.Count; k++)
        {
            var labels = etapes[k - 1];
            var vue = string.Join("   ", ordreArray.Take(k).Select(a => $"{a}={labels[a]}"));
            Console.WriteLine($"  étape {k} : {vue}");
        }

        return etapes[etapes.Count - 1];
    }

    public static bool EtiquettesEgales(Dictionary<string, string> gauche, Dictionary<string, string> droite)
    {
        return gauche.Count == droite.Count
            && gauche.All(entry => droite.TryGetValue(entry.Key, out var valeur) && valeur == entry.Value);
    }

    // Deux étiquetages ont-ils les mêmes clés et les mêmes valeurs ?
    public static bool LabelsEgaux(
        Dictionary<string, string> attendu,
        Dictionary<string, string> calcule)
    {
        return attendu.Count == calcule.Count
            && attendu.All(entry => calcule.TryGetValue(entry.Key, out var valeur) && valeur == entry.Value);
    }

    public static bool EtapesEgales(
        List<Dictionary<string, string>> gauche,
        List<Dictionary<string, string>> droite)
    {
        return gauche.Count == droite.Count
            && gauche.Zip(droite).All(pair => EtiquettesEgales(pair.First, pair.Second));
    }
}

// Exercice 5 — Combien d'étiquetages finaux distincts selon l'ordre d'arrivée ?
//
// La trajectoire montre que l'étiquetage final ne dépend pas de l'ordre d'arrivée sur
// `af1`, mais que les trajectoires intermédiaires diffèrent. L'exercice demande de généraliser
// cette observation à tous les ordres possibles.
//
// Objectif : implémenter `EtiquetagesFinauxDistincts(af)` qui énumère toutes les permutations
// des arguments de `af`, recalcule l'étiquetage final pour chacune, puis renvoie le nombre
// d'étiquetages finaux distincts. Les helpers `TrajectoireDemo.Trajectoire` et
// `TrajectoireDemo.EtapesEgales` sont déjà fournis.
//
// Le stub ci-dessous reste volontairement sans solution : il doit être complété par l'apprenant.
public static class TrajectoireExerciseStub
{
    // TODO etudiant : énumérer toutes les permutations des arguments de `af`, recalculer
    // l'étiquetage final de chacune (aide : TrajectoireDemo.Trajectoire) et renvoyer le
    // nombre d'étiquetages finaux distincts.
    public static int EtiquetagesFinauxDistincts(AF af)
    {
        return 0;
    }
}

Ordre d'arrivée : d -> c -> a -> b


  étape 1 : d=IN


  étape 2 : d=IN   c=UNDEC


  étape 3 : d=IN   c=UNDEC   a=IN


  étape 4 : d=IN   c=UNDEC   a=UNDEC   b=UNDEC


Ordre d'arrivée : a -> b -> c -> d


... (10 outputs left out)

  ordre a->b->c->d — étape 1 : attendu [a=IN] -> CONFORME


  ordre a->b->c->d — étape 2 : attendu [a=UNDEC, b=UNDEC] -> CONFORME


  ordre a->b->c->d — étape 3 : attendu [a=UNDEC, b=UNDEC, c=UNDEC] -> CONFORME


  ordre a->b->c->d — étape 4 : attendu [a=UNDEC, b=UNDEC, c=UNDEC, d=IN] -> CONFORME


### Ce que la trajectoire montre

Les deux ordres convergent vers le **même** étiquetage final — $d$=IN, $c$=UNDEC, $a$ et
$b$=UNDEC — mais par des chemins différents. Dans l'ordre $a \to b \to c \to d$, l'argument
$a$ est **IN** tant qu'il est seul, puis bascule en **UNDEC** dès que son attaquant $b$
arrive : l'acceptation est *révisée*, pas raffinée. C'est la non-monotonie, rendue visible
par le seul fait d'avoir regardé la suite plutôt que sa limite.

Deux remarques pour la suite :

- **$c$ est UNDEC, pas OUT.** L'argument auto-attaqué n'est rejeté par personne — il
s'exclut lui-même sans qu'un IN ne l'attaque. La règle « OUT $=$ attaqué par un IN »
ne s'applique donc pas à lui, et c'est bien ce que l'énoncé de l'Exercice 3 anticipe.
- **L'ordre est un paramètre, pas un détail.** Si la trajectoire dépend de l'ordre
d'arrivée, alors comparer deux protocoles de débat revient à comparer deux trajectoires
sur le *même* AF — ce que l'étiquetage final, à lui seul, ne permet pas.

### Lien avec le substrat 2025-EPITA-IS

Cette construction existe sous forme outillée dans le dépôt *2025-Epita-Intelligence-Symbolique*,
en `abs_arg_dung/aif_adapter.py` : `aif_labelling_trajectory` construit la même suite
d'étiquetages à partir d'un flux d'arguments et d'attaques **AIF** (undercut / undermine /
rebut), avec `aif_label_transitions` pour extraire les basculements et `render_aif_trajectory`
pour l'affichage. Chaque étape y est vérifiée contre les backends Dung (natif et TweetyProject)
qui se servent mutuellement d'oracle.

Le vocabulaire de sortie est le même à la casse près (`in` / `out` / `undec`), de sorte que
les étiquetages produits ici et là-bas se comparent directement. La cellule ci-dessus reste
volontairement **autonome** — C# standard (BCL .NET seule), aucune dépendance — pour que le notebook
s'exécute sans installer quoi que ce soit.

Une conséquence de cette autonomie mérite d'être dite, pour que le motif de démo ne soit
pas pris pour l'algorithme de production : `Trajectoire` recalcule `Grounded` **de zéro**
à chaque arrivée, et `Grounded` reconstruit son point fixe par itérations successives ; le coût
reste donc linéaire en le nombre d'itérations du point fixe, mais l'énumération générale des
sous-ensembles n'est pas employée par `Grounded`. C'est sans conséquence sur les quatre arguments
canoniques, et rédhibitoire sur un flux réel. Le substrat 2025-EPITA-IS procède par
**propagation incrémentale** des étiquettes : il met à jour l'étiquetage à partir du
précédent au lieu de le recalculer, ce qui est précisément ce que la structure de
trajectoire rend possible.

## À retenir

Ce notebook a reconstruit, de zéro et en C# standard (BCL .NET seule), les trois sémantiques
fondatrices de l'argumentation abstraite de Dung :

- **Grounded** — l'unique extension complète minimale, attitude sceptique.
- **Preferred** — les extensions complètes maximales, attitude crédule (potentiellement
  multiples et incompatibles).
- **Stable** — les extensions admissibles qui statuent sur tout argument extérieur ;
  **non garantie d'existence**.

Le cas canonique $\langle \{a,b,c,d\}, \{(a,b),(b,a),(c,c)\} \rangle$ a montré les trois
verdicts diverger simultanément : grounded $= \{d\}$, preferred $= \{\{a,d\}, \{b,d\}\}$,
stable inexistante. C'est la richesse structurelle qui justifie l'existence de trois
sémantiques plutôt qu'une seule : aucune ne capte à elle seule toutes les nuances de
l'acceptabilité argumentative.

Le § 4bis complète ce tableau : la **semi-stable** (Caminada 2006) répare le défaut
constaté — elle existe toujours et coïncide avec la stable quand celle-ci existe — et
l'**idéale** (Dung, Mancarella & Toni 2007) fournit le verdict unique sceptique
$\text{grounded} \subseteq \text{idéale} \subseteq \bigcap\text{preferred}$, celui
qu'exige un pipeline qui doit trancher.

## Vérifiez votre compréhension

**Question 1 — Sur l'AF canonique ⟨{a,b,c,d}, {(a,b),(b,a),(c,c)}⟩, pourquoi grounded = {d} alors que preferred contient {a,d} et {b,d} ?**

La cellule de la section 4 affiche `grounded : {d}` et `preferred : ['{b,d}', '{a,d}']`.

<details><summary>Réponse</summary>

Parce que `c` s'auto-attaque : il ne peut jamais être accepté et n'apparaît donc dans aucune extension. `d` est inattaqué et entre dans la grounded dès la première itération. `a` et `b` s'attaquent mutuellement sans qu'un tiers ne les défende : la grounded refuse de trancher et s'arrête à `{d}`. Les extensions preferred sont maximales : chacune prend parti, d'où `{a,d}` et `{b,d}` comme verdicts crédules incompatibles.

</details>

**Question 2 — Pourquoi la semi-stable est-elle appelée « réparation » de la stable ? Que se passe-t-il sur af3 = ⟨{a,b,c}, {a->b}⟩ ?**

La cellule de la section 4bis affiche l'absence de stable sur `af1`, puis, sur `af3`, `stable : ['{a,c}']`, `semi-stable : ['{a,c}']` et `coïncidence semi-stable == stable (setwise) : True`.

<details><summary>Réponse</summary>

La stable n'existe pas toujours ; `af1` en est le contre-exemple. La semi-stable existe toujours et coïncide avec la stable dès qu'elle existe. Sur `af3`, l'attaque `a->b` conduit à l'extension stable `{a,c}`, également semi-stable et preferred.

</details>

**Question 3 — Que montrent les deux témoins de l'encadrement ground ⊆ idéale ⊆ ∩preferred ?**

Sur `af_id1`, la grounded est vide tandis que l'idéale et l'intersection valent toutes deux `{x,y}`. Sur `af_id2`, la grounded et l'idéale valent toutes deux `{}`, tandis que l'intersection vaut `{z}`.

<details><summary>Réponse</summary>

`af_id1` montre une défense mutuelle qui permet à l'idéale de dépasser la grounded, tandis que l'intersection preferred coïncide encore avec l'idéale. `af_id2` montre le cas inverse : `z` appartient à toutes les extensions preferred, mais il n'est pas défendable sans choisir entre `a` et `b` ; l'idéale retombe donc sur l'ensemble vide. Les deux témoins situent les extrémités possibles de l'encadrement.

</details>

**Question 4 — Les deux ordres d'arrivée de la trajectoire d'étiquetage convergent-ils ? Et alors, pourquoi les garder tous les deux ?**

La cellule « trajectoire » affiche `Étiquetage FINAL identique ? True` et `Trajectoires identiques ? False`.

<details><summary>Réponse</summary>

Ils convergent vers le même étiquetage final : $d$=IN, $c$=UNDEC, $a$ et $b$=UNDEC. Les trajectoires diffèrent pourtant : dans l'ordre `a → b → c → d`, `a` passe par IN avant de devenir UNDEC ; dans l'ordre `d → c → a → b`, `a` passe lui aussi par IN à l'étape 3, puis devient UNDEC lorsque `b` arrive. La trajectoire conserve l'ordre des révisions, que le seul verdict final ne montre pas.

</details>

## Attestation de parité

Après exécution intégrale sur le noyau `.net-csharp`, les contrôles exécutables des sections 1 à 4bis et de la trajectoire ont vérifié, valeur par valeur, les résultats recalculés du port C# : attaquants de `a`, les cas de conflit, les résultats de défense et de défaite, les quatre cas d'admissibilité, les extensions exactes de `af3` (`{a,c}` pour stable, semi-stable et preferred), les valeurs exactes des deux témoins de l'idéale (preferred, grounded, idéale et ∩preferred), l'encadrement `grounded ⊆ idéale ⊆ ∩preferred`, le contrôle source « les trois ensembles ne sont pas tous égaux » et les étiquettes attendues à chaque étape des deux ordres d'arrivée. Chaque attendu est affiché avec son statut `CONFORME` ou `ÉCART`, et toute divergence lève une exception. Les cinq exercices restent des stubs compilables avec `// TODO etudiant`, sans solution révélée.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place des sémantiques de Dung dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Utiliser le solveur** : [Tweety-5](../Tweety/Tweety-05-Abstract-Argumentation-Python.ipynb) applique ces sémantiques via TweetyProject (JVM) sur des AF plus grands, où l'énumération exhaustive devient prohibitive. Le notebook [2-formal §6](Argumentation-05-Formal-Verification-Python.ipynb) en fait usage dans le pipeline agentique.

**Prouver les théorèmes** : [Tweety-05b-Lean](../Tweety/Tweety-05b-Argumentation-Lean.ipynb) formalise en Lean les inclusions (grounded ⊆ preferred) et les conditions d'existence de la sémantique stable.

**Argumentation structurée** : le formalisme abstrait de Dung ignore le contenu des arguments. ASPIC+, ABA et DeLP (accessibles via Tweety) reconstruisent la relation d'attaque à partir d'une logique sous-jacente — la série [Tweety](../Tweety/) les explore.
